# Learnability of the charge from <Z> and <XX> over the (p, h) plane

Issue #179. The experiment is `experiments/QELM_Q/exp_QELM_Q.py` of `04_QELM_Q_measurements.ipynb` (issue #163), swept this time over a 2D grid of the measurement probability `p` and the transverse field `h` (0 to 3 J), for two sizes `N = 4` and `N = 6`.

The reservoir is a disordered TFIM `H = h sum_i Z_i + sum_{i<j} J_ij X_i X_j` on the `Ns = N - 1` system qubits (`J_ij` uniform in `[-J/2, J/2]`, `J = 1`), with an ancilla `R` entangled with the charge of the system, `(|0>_R |Q1>_S + |1>_R |Q2>_S) / sqrt(2)` with `Q1 = Ns // 2` and `Q2 = Ns // 2 - 1`. For each sample:

1. `R` is measured in Z first: the outcome fixes the charge sector of the system, and the label is that charge, `Y = Q` (`Q1` or `Q2`, probability 1/2 each).
2. The system evolves for the total time `t = 10 / J` in 5 steps, and **between** two steps each system qubit is measured projectively in Z with probability `p`. The outcomes are not features of the first readouts: they only disturb the state. The last section adds them to the input.
3. Two readouts of `Q` are fitted, each on one set of expectation values of the final state, plus a bias:
   - **`<Z_k>`**, the one-site magnetizations of the system qubits (`Ns` features);
   - **`<X_i X_j>`**, the two-site correlators over the pairs `i < j` of system qubits (`Ns (Ns - 1) / 2` features).

Each readout is a ridge regression on `Q`, scored on a test set by two metrics:

- the **accuracy**: a prediction is right when it lies on the side of `(Q1 + Q2) / 2` of the true charge, 0.5 for a readout that learns nothing;
- the **R2** of the prediction of `Q`, its continuous counterpart: 1 for a perfect prediction, 0 for the mean, negative when the readout does worse than the mean.

The observables of the issue, `<Z>` and `<XX>`, are read here as the **features** of the readout, and the learnability is that of the charge `Q` they carry: an accuracy needs a two-valued target, and `Q` is the one of the experiment. The other feature sets of `exp_QELM_Q` (`<X_k>`, `<Y_k>`, `<Y_i Y_j>`, `<Z_i Z_j>`, `<X_i Y_j>`) are recorded by every run but not drawn here; notebook 04 shows them.

### What to expect

- The `X_i X_j` couplings change `Q` by 0 or 2, and a Z measurement commutes with `Q`: the parity of `Q` is always conserved, `Q` itself only when `h >> J`, where the field term dominates.
- **`<Z_k>`** is diagonal in the Z basis, so the Z measurements do not erase what it holds about `Q`. At large `h` the charge is nearly conserved and `sum_k <Z_k> = Ns - 2 Q`: the readout should be accurate whatever `p`. At small `h` the couplings move the charge around and `<Z_k>` no longer adds up to `Q`.
- **`<X_i X_j>`** keeps the parity but is nonzero only through coherences between Z basis states, which the Z measurements destroy: this is the observable for which the measurements should cost accuracy as `p` grows. Notebook 04 found that at `N = 6` and `t = 10 / J` the random couplings already dephase these coherences at `p = 0` (its last section, where a shorter time `t = 0.5 / J` recovers them).

In [ ]:
import os
from pathlib import Path

# Jupyter starts in the notebook's folder. Move to the repository root (the
# folder that contains pyproject.toml) so conf/, experiments/ and outputs/
# resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Parameters

Each value can be changed here, or through the environment variable of the same name (this is what the test `tests/test_qelm_q_p_h_notebook.py` does for a small sweep). Notation, the same in every notebook: `N` is the total number of qubits, ancilla included, and `L` the number of samples per run (`samples.L`). `H_VALUES` are the transverse fields `TFIM.h`, in units of `TFIM.J = 1`, from 0 to 3, where the learnability changes (above `h ≈ 2 J` the readout on `<Z_k>` is near perfect at every `p`); `T` is the total evolution time `evolution.t`, in units of `1 / J` (10 is the config default). `FIXED_P` are the values of `p` drawn in the figure against `h`; each must be one of `P_VALUES`.

The sweep goes to this notebook's own folder, `outputs/QELM_Q_p_h/<date_time>/`, and the notebook reads back exactly that folder (issue #151). `RUN = False` reads back the latest sweep of this folder without launching a new one.

The default grid has `2 N x 10 h x 11 p x 5 seeds = 1100` runs. Each run uses a single thread (`OMP_NUM_THREADS=1` below): its matrices are at most 64 x 64, too small for threads to help, and with the default thread count a run at `N = 6` was measured about 6 times slower (25 s instead of 4 s). The whole sweep takes about an hour on one core.

In [ ]:
N_VALUES = os.environ.get("N_VALUES", "4 6")  # total qubits N, ancilla included
H_VALUES = os.environ.get(
    "H_VALUES", "0 0.25 0.5 0.75 1 1.25 1.5 2 2.5 3"
)  # TFIM.h / J
P_VALUES = os.environ.get("P_VALUES", "0.0 0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1.0")
SEEDS = os.environ.get("SEEDS", "42 43 44 45 46")
T = os.environ.get("T", "10")  # evolution.t, in units of 1 / J
L = os.environ.get("L", "400")  # samples per run
FIXED_P = os.environ.get("FIXED_P", "0.0 0.2 0.5 0.8 1.0")  # curves against h
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "1") == "1"

## Launch the sweep

The launcher `experiments/QELM_Q/exp_QELM_Q.sh` runs every combination of `N`, `h`, seed and `p`, measurements between steps only (`FINAL_VALUES=false`). Each run lands in `N<N>_h<h>_t<t>_finalfalse_seed<seed>_p<p>/` of the sweep folder.

In [ ]:
from QELM.utils import latest_sweep, new_sweep_dir, run_with_progress

# This notebook's own output folder (issue #151).
EXPERIMENT = "QELM_Q_p_h"

if RUN:
    # Create the sweep folder here and hand it to the launcher (OUT_DIR):
    # the cells below read this sweep and no other.
    sweep_dir = new_sweep_dir(OUT_ROOT, EXPERIMENT)
    env = {
        **os.environ,
        "N_VALUES": N_VALUES,
        "H_VALUES": H_VALUES,
        "T_VALUES": T,
        "FINAL_VALUES": "false",
        "SEEDS": SEEDS,
        "P_VALUES": P_VALUES,
        "L": L,
        "OUT_DIR": str(sweep_dir),
        # One thread per run: the matrices are too small for threads to help,
        # and they slow a run at N = 6 down about 6 times.
        "OMP_NUM_THREADS": "1",
        "OPENBLAS_NUM_THREADS": "1",
        "MKL_NUM_THREADS": "1",
    }
    n_runs = (
        len(N_VALUES.split())
        * len(H_VALUES.split())
        * len(SEEDS.split())
        * len(P_VALUES.split())
    )
    # One bar for the whole sweep, one step per run; raises
    # CalledProcessError, with the runs' output, if a run fails.
    run_with_progress(
        ["bash", "experiments/QELM_Q/exp_QELM_Q.sh"], total=n_runs, env=env
    )
else:
    # RUN = False: the latest sweep of this notebook only.
    sweep_dir = latest_sweep(OUT_ROOT / EXPERIMENT)
print(f"sweep: {sweep_dir}")

## Read back the metrics

Every metric is averaged over the seeds, with the standard error `std / sqrt(n_runs)`. The table prints, for each `(N, h, p)`, the accuracy and the R2 of the readouts on `<Z_k>` and `<X_i X_j>`, and the chance level `accuracy_baseline` (always predicting the majority charge of the training set: a finite test set is never exactly balanced, issue #130). The readouts on the record, last section, are read in the same table.

In [ ]:
from QELM.utils import load_sweep, summarize

# (feature set in exp_QELM_Q, label): the observables of the final state,
# then the record M of the intermediate measurements, alone and next to them.
OBSERVABLES = [("Z", "<Z_k>"), ("XX", "<X_i X_j>")]
RECORD_OBSERVABLES = [
    ("M", "M"),
    ("M_Z", "[M, <Z_k>]"),
    ("M_XX", "[M, <X_i X_j>]"),
]
# (score, label, bottom and top of the heatmap color scale).
SCORES = [
    ("accuracy", "accuracy on Q (test)", 0.5, 1.0),
    ("r2", "R2 of Q (test)", 0.0, 1.0),
]
KEYS = [f"{score}_{name}" for name, _ in OBSERVABLES for score, *_ in SCORES]
RECORD_KEYS = [
    f"{score}_{name}" for name, _ in RECORD_OBSERVABLES for score, *_ in SCORES
]
rows = load_sweep(
    sweep_dir,
    keys=(
        ("N", "N"),
        ("h", "TFIM.h"),
        ("seed", "seed"),
        ("p", "projective_measurement.p"),
    ),
)
table = summarize(
    rows,
    by=("N", "h", "p"),
    metrics=[*KEYS, *RECORD_KEYS, "accuracy_baseline", "S_half_mean"],
)


def print_table(keys):
    # One line per (N, h, p): mean ± standard error of each key.
    print(
        f"{'N':>3} {'h':>5} {'p':>5} {'runs':>4} "
        + " ".join(f"{key:>17}" for key in keys)
    )
    for r in table:
        values = " ".join(
            f"{r[f'{key}_mean']:>7.3f} ± {r[f'{key}_sem']:<7.3f}" for key in keys
        )
        print(f"{r['N']:>3} {r['h']:>5g} {r['p']:>5.2f} {r['n_runs']:>4} {values}")


print(f"{len(rows)} runs")
print_table([*KEYS, "accuracy_baseline"])

## Metrics against p, one curve per field h

One row per size `N`, one column per observable and score: the accuracy and the R2 of `<Z_k>`, then of `<X_i X_j>`. One curve per field `h`, from dark (`h = 0`) to light (the largest `h`), the mean over the seeds with the standard error as error bars. On the accuracy panels the dotted gray line is chance, 0.5; on the R2 panels, 0.

The three plotting functions defined here are used again for the readouts on the record, in the last section. The figure and the data it shows are saved in the sweep folder, as `metrics_vs_p.png` and `metrics_vs_p.json` (`QELM.utils.plots.save_figure`).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import colormaps

from QELM.utils.plots import (
    draw_curve,
    draw_heatmap,
    save_figure,
    sizes,
    subplots,
    sweep_curve,
    sweep_grid,
)

Ns = sizes(table)
hs = sorted({r["h"] for r in table})
ps = sorted({r["p"] for r in table})
# An ordered parameter gets an ordered color scale.
h_colors = {h: colormaps["viridis"](k / max(len(hs) - 1, 1)) for k, h in enumerate(hs)}
fixed_p = [float(p) for p in FIXED_P.split()]
missing = [p for p in fixed_p if p not in ps]
assert not missing, f"FIXED_P not in the sweep: {missing}"
p_colors = {
    p: colormaps["plasma"](0.85 * k / max(len(fixed_p) - 1, 1))
    for k, p in enumerate(fixed_p)
}


def panels(observables):
    # One panel per observable and score: (name, label, score, score label).
    return [
        (name, obs, score, label)
        for name, obs in observables
        for score, label, *_ in SCORES
    ]


def chance(ax, score):
    # The level of a readout that learns nothing.
    if score == "accuracy":
        ax.axhline(0.5, color="gray", linestyle=":", label="chance")
    else:
        ax.axhline(0.0, color="gray", linestyle=":")


def plot_curves(observables, x_key, figure_name, title):
    # Rows: N; columns: observable x score. Against p, one curve per h;
    # against h, one curve per p of FIXED_P.
    cols = panels(observables)
    fig, axes = subplots(
        len(Ns), len(cols), figsize=(5.5 * len(cols), 4.8 * len(Ns)), squeeze=False
    )
    if x_key == "p":
        curves = [(h, "h", h_colors[h], f"h = {h:g} J") for h in hs]
        xlabel = "measurement probability p"
    else:
        curves = [(p, "p", p_colors[p], f"p = {p:g}") for p in fixed_p]
        xlabel = "transverse field h (units of J)"
    for N, row in zip(Ns, axes, strict=True):
        for (name, obs, score, label), ax in zip(cols, row, strict=True):
            for value, key, color, legend in curves:
                rows_v = [r for r in table if r[key] == value]
                draw_curve(
                    ax,
                    sweep_curve(rows_v, N, x_key, f"{score}_{name}"),
                    color,
                    label=legend,
                )
            chance(ax, score)
            ax.set(xlabel=xlabel, ylabel=label, title=f"{obs}: {score}, N = {N}")
        row[0].legend(fontsize="small", ncol=2 if x_key == "p" else 1)
    fig.suptitle(title)
    path = save_figure(fig, sweep_dir, figure_name, data={"table": table})
    print(f"figure: {path}")
    plt.show()
    return path, axes


def plot_heatmaps(observables, figure_name, title):
    # Rows: observable x score; columns: N. p on x, h on y.
    rows_ = panels(observables)
    fig, axes = subplots(
        len(rows_), len(Ns), figsize=(6.0 * len(Ns), 4.8 * len(rows_)), squeeze=False
    )
    grids = {}
    for (name, obs, score, label), row in zip(rows_, axes, strict=True):
        vmin, vmax = next((lo, hi) for s, _, lo, hi in SCORES if s == score)
        for N, ax in zip(Ns, row, strict=True):
            grid = sweep_grid(table, "p", "h", f"{score}_{name}", size=N)
            grids[(N, name, score)] = grid
            draw_heatmap(ax, grid, label=label, vmin=vmin, vmax=vmax)
            ax.set(
                xlabel="measurement probability p",
                ylabel="transverse field h (units of J)",
                title=f"{obs}: {score}, N = {N}",
            )
    fig.suptitle(title)
    path = save_figure(fig, sweep_dir, figure_name, data={"table": table})
    print(f"figure: {path}")
    plt.show()
    return path, axes, grids


figure_p, axes_p = plot_curves(
    OBSERVABLES,
    "p",
    "metrics_vs_p",
    "Learnability of Q from <Z_k> and <X_i X_j> against p, for each field h",
)

## Metrics against h, for fixed values of p

The same panels with the field `h` on the x axis, one curve per value of `p` in `FIXED_P`, from dark (`p = 0`) to light (`p = 1`). This is the cut along the field axis of the issue: the change of the curves with `h` at fixed `p` shows where the field takes over the couplings.

Saved in the sweep folder as `metrics_vs_h.png` and `metrics_vs_h.json`.

In [ ]:
figure_h, axes_h = plot_curves(
    OBSERVABLES,
    "h",
    "metrics_vs_h",
    "Learnability of Q from <Z_k> and <X_i X_j> against h, for fixed p",
)

## Heatmaps over the (p, h) plane

One color map per observable and score (rows) and per size (columns): `p` on the x axis, `h` on the y axis, the mean over the seeds as the color, one cell per point of the grid (`QELM.utils.plots.draw_heatmap`). The accuracy scale runs from chance, 0.5, to 1; the R2 scale from 0 to 1, a negative R2 being drawn in the color of 0. Comparing the two columns shows the finite-size effects.

Saved in the sweep folder as `heatmaps_p_h.png` and `heatmaps_p_h.json`.

In [ ]:
heatmap_path, axes_maps, grids = plot_heatmaps(
    OBSERVABLES, "heatmaps_p_h", "Learnability of Q over the (p, h) plane"
)

## Readout on the record and the observables, phi = [M, X]

The same runs, with the input of the ridge readout extended by the **record of the intermediate measurements** `M`: for each sample, the outcomes of the `nbr_time_evolution - 1 = 4` rounds on the `Ns` system qubits, +1 or -1 for a measured qubit and 0 for an unmeasured one, as in `experiments/QELM_M/exp_QELM_M.py`. `M` is the dynamic input, read during the evolution; `X` is the static input of the standard QELM, read on the final state. Three readouts, written by every run of `exp_QELM_Q` (issue #179):

- **`M`** alone, the reference: what the record alone tells about `Q`;
- **`[M, <Z_k>]`**: `phi = [M, X]` with `X` the one-site `<Z_k>`;
- **`[M, <X_i X_j>]`**: `phi = [M, X]` with `X` the two-site `<X_i X_j>`.

At `p = 0` the record is all zeros: `M` alone is at chance and the two others are exactly the readouts on `<Z_k>` and `<X_i X_j>` above. As `p` grows the record holds more outcomes. At large `h`, where `Q` is nearly conserved, the outcomes of a full round add up to `Ns - 2 Q`, so `M` alone should learn `Q` near `p = 1`; the question is whether the record makes up for what the measurements remove from `X`, where `X` alone fails (small `h`, or `<X_i X_j>` at any `p > 0`).

The three figures of the sections above, for these readouts: `metrics_vs_p_record.png`, `metrics_vs_h_record.png` and `heatmaps_p_h_record.png` in the sweep folder.

In [ ]:
print_table([*RECORD_KEYS, "accuracy_baseline"])
record_p, record_axes_p = plot_curves(
    RECORD_OBSERVABLES,
    "p",
    "metrics_vs_p_record",
    "Learnability of Q from phi = [M, X] against p, for each field h",
)

In [ ]:
record_h, record_axes_h = plot_curves(
    RECORD_OBSERVABLES,
    "h",
    "metrics_vs_h_record",
    "Learnability of Q from phi = [M, X] against h, for fixed p",
)

In [ ]:
record_maps, record_axes_maps, record_grids = plot_heatmaps(
    RECORD_OBSERVABLES,
    "heatmaps_p_h_record",
    "Learnability of Q from phi = [M, X] over the (p, h) plane",
)

## What the default sweep shows

The default sweep (1100 runs, `h` from 0 to 3 J, 5 seeds, `L = 400`, `t = 10 / J`), run on 2026-10-22. The numbers are means over the seeds; the chance level `accuracy_baseline` is 0.54 at `N = 4` and 0.49 at `N = 6`.

**`<Z_k>` alone.** At `p = 0` the charge is read perfectly from `h = 0.5 J` up (accuracy 1.00 at `N = 4`, 0.95 at `N = 6` and `h = 0.5 J`), and at chance at `h = 0`. The measurements cost accuracy, more the smaller the field and the larger the system: at `h = J` and `p = 1`, 0.94 at `N = 4` and 0.79 at `N = 6`; at `h = 0.5 J`, 0.65 and chance. The R2 shows the same boundary further up: at `h = 2 J` and `p = 1`, 0.57 (`N = 4`) and 0.54 (`N = 6`). In the heatmaps the line where learning fails rises in `h` as `p` grows, and higher for `N = 6`.

**`<X_i X_j>` alone.** Perfect at `h = 0, p = 0`, where `H = sum J_ij X_i X_j` commutes with every `X_i X_j`, so that these correlators keep their initial values, which depend on `Q`. One round of measurements removes this. For `h > 0`, about 0.73 to 0.82 at `p = 0` for `N = 4`, chance by `p ≈ 0.3`; at `N = 6`, chance everywhere but `h = 0, p = 0` (dephasing by the couplings at `t = 10 / J`, as in notebook 04).

**The record `M` alone** learns `Q` more as `p` grows, and the more the larger the field: at `p = 1`, 0.98 to 1.00 for `h ≥ J` at both sizes, but 0.85 (`N = 4`) and 0.63 (`N = 6`) at `h = 0.5 J`, and chance at `h = 0`. The outcomes of a round add up to `Ns - 2 Q` only when the charge is conserved; at small `h` the couplings move it between the rounds.

**`phi = [M, <Z_k>]`** is as good as the better of its two parts, and better than both at intermediate `p`: the observables carry `Q` at small `p`, the record at large `p`. The dip of `<Z_k>` alone at large `p` disappears: at `h = J`, `N = 6`, `p = 1`, 0.98 instead of 0.79. At `p = 1` the readout is the one of `M` alone (same scores), the final `<Z_k>` adding nothing to a complete record. What remains hard is small fields and intermediate `p`, where neither part holds the charge: at `h = 0.5 J`, `N = 6`, `p = 0.5`, 0.58.

**`phi = [M, <X_i X_j>]`** is the readout of `M` alone, plus the `h = 0, p = 0` point and, at `N = 4`, the accuracy of `<X_i X_j>` at small `p`: the correlators add nothing once the measurements have erased the coherences.

## Finite-size scaling: N = 8, 10, 12 at h = J and 2 J

Issue #183 and its review on PR #184. Is there a measurement-induced transition in the learnability of `Q` at a critical `p_c`, and where? Near `p_c` the curves of the larger systems should get steeper and cross, which locates `p_c`. A second sweep, separate from the `(p, h)` grid:

- **sizes `N = 8, 10, 12`**, ancilla included, so `Ns = 7, 9, 11` system qubits: `N = 4` and `6` are too far from the scaling regime (the first version of this section, with `N = 4` to `10`, showed it);
- **`p` every 0.05**, 21 points from 0 to 1 (`np.linspace(0, 1, FSS_P_POINTS)`), to resolve the crossing;
- **two fields, `h = J` and `h = 2 J`**, to see how `p_c` moves with the field;
- **one readout, `phi = [M, <Z_k>]`**, the record of the measurements next to the final magnetizations, which shows the learnability transition; `<Z_k>` and `<X_i X_j>` alone are left out. Every run still writes all its readouts.

The rest is the experiment above: `t = 10 / J` in 5 steps, measurements between steps only, the same seeds and `L`.

**Cost.** A density matrix of `N = 12` qubits is `4096 x 4096`. The state stays pure (the ancilla is measured first, the evolution is unitary, a projective measurement keeps a pure state pure), so this sweep follows the **state vector** instead (`state_representation=vector` in `exp_QELM_Q`, `STATE=vector` for the launcher): `exp(-i H dt) psi` from the sparse `H`, without the dense propagator. For a given seed it gives the same features as the density matrix, to `1e-14` (`tests/test_exp_QELM_Q.py`), about 20 times faster at `N = 10`. Measured on one core: 0.03 s per sample at `N = 8`, 0.06 s at `N = 10`, 0.23 s at `N = 12`, so a run of `L = 400` samples takes 15 s, 25 s and 90 s. The default sweep has `3 N x 2 h x 21 p x 5 seeds = 630` runs, about 7.5 hours on one core. The launcher runs `FSS_JOBS` runs at once (`JOBS`), one core each, by default as many as the machine has cores.

Its own parameters, each with an environment variable of the same name. `FSS_RUN` defaults to `RUN`: with `FSS_RUN = 0` the notebook reads back the latest sweep of its folder, `outputs/QELM_Q_N_scaling/<date_time>/`.

In [ ]:
import numpy as np

FSS_N_VALUES = os.environ.get("FSS_N_VALUES", "8 10 12")  # total qubits N
FSS_H_VALUES = os.environ.get("FSS_H_VALUES", "1 2")  # TFIM.h / J
FSS_P_POINTS = int(os.environ.get("FSS_P_POINTS", "21"))  # p in linspace(0, 1, .)
FSS_SEEDS = os.environ.get("FSS_SEEDS", SEEDS)
FSS_L = os.environ.get("FSS_L", L)
FSS_JOBS = os.environ.get("FSS_JOBS", str(os.cpu_count() or 1))  # runs at once
FSS_RUN = os.environ.get("FSS_RUN", "1" if RUN else "0") == "1"
FSS_P_VALUES = " ".join(f"{p:g}" for p in np.linspace(0.0, 1.0, FSS_P_POINTS))
FSS_EXPERIMENT = "QELM_Q_N_scaling"
# The readout of this section: the record next to <Z_k>, phi = [M, <Z_k>].
FSS_NAME, FSS_LABEL = "M_Z", "[M, <Z_k>]"
FSS_KEYS = [f"{score}_{FSS_NAME}" for score, *_ in SCORES]

if FSS_RUN:
    fss_dir = new_sweep_dir(OUT_ROOT, FSS_EXPERIMENT)
    fss_env = {
        **os.environ,
        "N_VALUES": FSS_N_VALUES,
        "H_VALUES": FSS_H_VALUES,
        "T_VALUES": T,
        "FINAL_VALUES": "false",
        "SEEDS": FSS_SEEDS,
        "P_VALUES": FSS_P_VALUES,
        "L": FSS_L,
        "OUT_DIR": str(fss_dir),
        # The state vector: same results as rho, affordable at N = 12.
        "STATE": "vector",
        "JOBS": FSS_JOBS,
        "OMP_NUM_THREADS": "1",
        "OPENBLAS_NUM_THREADS": "1",
        "MKL_NUM_THREADS": "1",
    }
    fss_runs = (
        len(FSS_N_VALUES.split())
        * len(FSS_H_VALUES.split())
        * len(FSS_SEEDS.split())
        * FSS_P_POINTS
    )
    # With JOBS > 1 the bar moves when a run starts rather than when it ends.
    run_with_progress(
        ["bash", "experiments/QELM_Q/exp_QELM_Q.sh"], total=fss_runs, env=fss_env
    )
else:
    fss_dir = latest_sweep(OUT_ROOT / FSS_EXPERIMENT)
print(f"sweep: {fss_dir}")

fss_rows = load_sweep(
    fss_dir,
    keys=(
        ("N", "N"),
        ("h", "TFIM.h"),
        ("seed", "seed"),
        ("p", "projective_measurement.p"),
    ),
)
fss_table = summarize(
    fss_rows, by=("N", "h", "p"), metrics=[*FSS_KEYS, "accuracy_baseline"]
)
print(f"{len(fss_rows)} runs")
print(
    f"{'N':>3} {'h':>5} {'p':>5} {'runs':>4} "
    + " ".join(f"{key:>17}" for key in [*FSS_KEYS, "accuracy_baseline"])
)
for r in fss_table:
    values = " ".join(
        f"{r[f'{key}_mean']:>7.3f} ± {r[f'{key}_sem']:<7.3f}"
        for key in [*FSS_KEYS, "accuracy_baseline"]
    )
    print(f"{r['N']:>3} {r['h']:>5g} {r['p']:>5.2f} {r['n_runs']:>4} {values}")

### Accuracy and R2 of [M, <Z_k>] against p, one curve per size

One figure: one row per field `h`, one column per score (accuracy, R2). One curve per size `N`, from dark (`N = 8`) to light (`N = 12`), the mean over the seeds with the standard error as error bars; the dotted gray line is chance (accuracy 0.5, R2 0). A crossing of the curves marks the `p_c` of a transition; curves that only shift with `N`, without crossing, mean no transition in this range of sizes.

Saved in the sweep folder as `metrics_vs_p_N.png` and `metrics_vs_p_N.json`.

In [ ]:
fss_Ns = sizes(fss_table)
fss_hs = sorted({r["h"] for r in fss_table})
N_colors = {
    N: colormaps["viridis"](0.85 * k / max(len(fss_Ns) - 1, 1))
    for k, N in enumerate(fss_Ns)
}
fig, axes_fss = subplots(
    len(fss_hs),
    len(SCORES),
    figsize=(5.5 * len(SCORES), 4.5 * len(fss_hs)),
    squeeze=False,
)
for h, row in zip(fss_hs, axes_fss, strict=True):
    rows_h = [r for r in fss_table if r["h"] == h]
    for (score, label, *_), ax in zip(SCORES, row, strict=True):
        for N in fss_Ns:
            draw_curve(
                ax,
                sweep_curve(rows_h, N, "p", f"{score}_{FSS_NAME}"),
                N_colors[N],
                label=f"N = {N}",
            )
        chance(ax, score)
        ax.set(
            xlabel="measurement probability p",
            ylabel=label,
            title=f"{FSS_LABEL}: {score}, h = {h:g} J",
        )
axes_fss[0, 0].legend(fontsize="small")
fig.suptitle(f"Learnability of Q from {FSS_LABEL}: finite-size scaling")
figure_fss = save_figure(fig, fss_dir, "metrics_vs_p_N", data={"table": fss_table})
print(f"figure: {figure_fss}")
plt.show()

### What the default sweep shows

The default sweep (630 runs, `N = 8, 10, 12`, `h = J` and `2 J`, `p` every 0.05, 5 seeds, `L = 400`, `t = 10 / J`, state vector), run on 2026-10-26. The numbers are means over the seeds; chance is 0.48 to 0.52.

**No crossing of the curves in this range of sizes.** At every `p` the larger system learns `Q` less well or as well, never better, so the sweep does not locate a `p_c` by a crossing and a data collapse is not possible yet.

**At `h = 2 J` there is a learnable phase at small `p`.** At `p = 0` the charge is read perfectly at every size (accuracy 1.00, R2 0.996 to 0.998), and the three curves coincide up to `p ≈ 0.3` for the accuracy (0.977, 0.958, 0.952 at `p = 0.3`) and `p ≈ 0.15` to `0.2` for the R2 (0.92 at `p = 0.15`). Beyond, the curves fan out and the larger systems drop further: at `p = 0.5`, accuracy 0.94, 0.84, 0.74 and R2 0.52, 0.28, 0.15 for `N = 8, 10, 12`. The size dependence switches on between `p ≈ 0.2` and `0.35`; this is the best estimate of the boundary these sizes give, and a transition there would leave the small-`p` side learnable as `N` grows and push the large-`p` side towards chance.

**At `h = J` the learnability falls with `N` already at `p = 0`**: accuracy 0.995, 0.965, 0.85 and R2 0.94, 0.80, 0.48 for `N = 8, 10, 12`. The field does not hold the charge against the couplings over `t = 10 / J` at these sizes, so there is no learnable phase to start from and the measurements only lower curves that are already falling: by `p = 0.4` the accuracy is below 0.6 at `N = 10` and `12`. Going from `h = 2 J` to `h = J` moves the boundary of the learnable region from `p ≈ 0.2`-`0.35` to `p = 0` or below at these sizes.

**The recovery at `p → 1`** (the record alone holds the charge when every qubit is measured) weakens with `N` at both fields: at `h = 2 J`, 0.98, 0.94, 0.885 at `p = 1`; at `h = J`, 0.78, 0.68, 0.59.

**Caveat on the number of samples.** The input of the readout has `5 Ns + 1` features (the record of 4 rounds on `Ns` qubits, the `Ns` values `<Z_k>` and the bias): 36, 46 and 56 for `N = 8, 10, 12`, fitted on 280 training samples at `L = 400`. The negative R2 at `h = J` (down to -0.29 at `N = 12`) shows the ridge fitting noise, and part of the drop with `N` may come from the fixed `L`. Scaling `L` with the number of features (`FSS_L`) is the next check before a data collapse.